In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR  # FIX 3: Added LR scheduler

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


torch.manual_seed(42)
np.random.seed(42)

from google.colab import drive
drive.mount('/content/drive')


weights = ResNet50_Weights.IMAGENET1K_V1

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.08), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = val_transform

class TransformSubset(torch.utils.data.Dataset):
    """Wraps a Subset and applies a transform per split."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset = datasets.ImageFolder(root="/content/drive/MyDrive/Project 4/dataset")
dataset_size = len(full_dataset)
print(f"Total Dataset Size: {dataset_size}")

train_size = int(0.7 * dataset_size)
val_size   = int(0.15 * dataset_size)
test_size  = dataset_size - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_subset, val_subset, test_subset = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

# Wrap with correct transforms
train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset,   val_transform)
test_dataset  = TransformSubset(test_subset,  test_transform)

class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Number of Classes: {num_classes}")
print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

BATCH_SIZE = 32
PIN = device.type == "cuda"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=PIN)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN)

def calculate_metrics(y_true, y_pred, class_names):
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    precision = report["weighted avg"]["precision"]
    recall    = report["weighted avg"]["recall"]
    f1        = report["weighted avg"]["f1-score"]
    return precision, recall, f1


def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)  # FIX 7: savefig instead of show
    plt.close()
    print("Confusion matrix saved to confusion_matrix.png")

In [ ]:
# Training Function
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc="Training")

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100*correct/total:.2f}%"})

    return running_loss / len(loader), 100 * correct / total

In [ ]:
# Validation Function
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        progress_bar = tqdm(loader, desc="Validation")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100*correct/total:.2f}%"})

    return running_loss / len(loader), 100 * correct / total


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10, scheduler=None):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        print("\n" + "=" * 60)
        print(f"Epoch [{epoch+1}/{epochs}]")
        print("=" * 60)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc   = validate(model, val_loader, criterion, device)

        if scheduler:
            scheduler.step()  # Step LR after each epoch

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.2f}%")
        print(f"Val Loss   : {val_loss:.4f}   | Val Acc   : {val_acc:.2f}%")
        print(f"LR         : {current_lr:.6f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_resnet50_transfer_learning.pth")
            print("Best model saved!")

    print("\nTraining Complete!")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    return history

In [ ]:
# Evaluation Function
def evaluate_model(model, loader, criterion, class_names, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Testing"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss     = running_loss / len(loader)
    test_accuracy = 100 * correct / total
    precision, recall, f1 = calculate_metrics(all_labels, all_preds, class_names)

    print("\n" + "=" * 60)
    print("TEST RESULTS")
    print("=" * 60)
    print(f"Test Loss : {test_loss:.4f}")
    print(f"Accuracy  : {test_accuracy:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    plot_confusion_matrix(all_labels, all_preds, class_names)

In [ ]:

# Training History Visualization
def plot_training_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"],   label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss Curve")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"],   label="Validation Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy Curve")
    plt.legend()

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.close()
    print("Training history saved to training_history.png")

In [ ]:
# Build Transfer Learning Model
model = resnet50(weights=weights)

# Phase 1: Freeze all backbone layers — only train the head
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Phase 1: Train classifier head only
# CosineAnnealingLR scheduler added to improve convergence and generalization.
EPOCHS_PHASE1 = 10

criterion = nn.CrossEntropyLoss()
optimizer_phase1 = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler_phase1 = CosineAnnealingLR(optimizer_phase1, T_max=EPOCHS_PHASE1, eta_min=1e-5)

print("\n" + "=" * 60)
print("PHASE 1: Training classifier head (backbone frozen)")
print("=" * 60)

history_phase1 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase1,
    device=device,
    epochs=EPOCHS_PHASE1,
    scheduler=scheduler_phase1,
)

In [ ]:
plot_training_history(history_phase1)

In [ ]:
# Phase 2 — Fine-tune backbone with lower LR
# Unfreeze all layers and train end-to-end at a reduced LR.
EPOCHS_PHASE2 = 5

for param in model.parameters():
    param.requires_grad = True

optimizer_phase2 = optim.Adam([
    {"params": model.fc.parameters(),  "lr": 1e-3},   # head: higher LR
    {"params": [p for name, p in model.named_parameters()
                if "fc" not in name],  "lr": 1e-4},   # backbone: lower LR
])
scheduler_phase2 = CosineAnnealingLR(optimizer_phase2, T_max=EPOCHS_PHASE2, eta_min=1e-6)

print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning full network (backbone unfrozen)")
print("=" * 60)

history_phase2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase2,
    device=device,
    epochs=EPOCHS_PHASE2,
    scheduler=scheduler_phase2,
)

In [ ]:
plot_training_history(history_phase2)

In [ ]:
# Load Best Model & Evaluate
model.load_state_dict(
    torch.load("best_resnet50_transfer_learning.pth", map_location=device)
)

evaluate_model(
    model=model,
    loader=test_loader,
    criterion=criterion,
    class_names=class_names,
    device=device,
)